# Static letters only: the results on the 30 letters the system is meant to recognise

Eleven letters are signed with movement: آ ؤ ئ ج ش م ن ٹ ڑ ھ ہ. The pipeline classifies one held hand
shape, so it cannot represent them. On 15 Nov 2026 Hassan and Abdullah decided to keep this as a stated
limitation of the project: the system covers the **30 static letters**, which are also the 30 letters with
calibration takes.

`train_val_test_models.ipynb` scored everything on all 40 dataset letters. This notebook repeats its two
splits with the dynamic letters removed from training and from testing (`splits.static_only`), so the
numbers match what the project now claims.

* **A. New person**: whole signers held out. The dataset signers have 29 of the 30 static letters (ء was only
  ever recorded in calibration).
* **B. Calibrated user**: whole calibration takes held out, 30 letters. The test takes are the same takes as in
  the earlier notebook; what changes is that the model no longer has the 11 dynamic classes to confuse them with.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, os, time
import numpy as np, pandas as pd
from scipy import stats
import evaluation as ev, splits, models, letter_models as lm

pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200)
P0, M0 = ev.load_dataset(include_users=True)              # everything, 41 letters
P, M = splits.static_only(P0, M0)                         # the 30 static letters
y, sg = M.letter.to_numpy(), M.signer.to_numpy()
user = splits.is_user(sg); D = ~user
yD, sD = y[D], sg[D]
OUT = "results/2026-11-15_static_letters"; os.makedirs(OUT, exist_ok=True)
OLD = "results/2026-11-15_train_val_test"
print(f"all letters:    {len(M0)} rows, {M0.letter.nunique()} letters")
print(f"static letters: {len(M)} rows ({D.sum()} dataset holds, {user.sum()} calibration takes), "
      f"{len(set(y))} letters, {len(set(yD))} of them in the dataset signers")

all letters:    1636 rows, 41 letters
static letters: 1319 rows (1028 dataset holds, 291 calibration takes), 30 letters, 29 of them in the dataset signers


## 1. What is removed

In [2]:
u0 = splits.is_user(M0.signer); dyn0 = M0.letter.isin(splits.DYNAMIC_LETTERS).to_numpy()
n_dyn = (dyn0 & ~u0).sum()
print(f"dataset holds labelled with a dynamic letter: {n_dyn} of {(~u0).sum()} ({100*n_dyn/(~u0).sum():.0f}%)")
print(M0[dyn0 & ~u0].letter.value_counts().rename("holds").to_frame().T.to_string())
print("\nstatic dataset holds per signer:", pd.Series(sD).value_counts().sort_index().to_dict())

W = json.load(open("word_letter_mapping.json", encoding="utf-8"))
hit = {w: [l for l in ls if l in splits.DYNAMIC_LETTERS] for w, ls in W.items()}
n_tok = sum(len(ls) for ls in W.values()); n_dtok = sum(len(v) for v in hit.values())
print(f"\nthe 54 dictionary words: {sum(bool(v) for v in hit.values())} contain at least one dynamic letter; "
      f"{n_dtok} of {n_tok} letter positions ({100*n_dtok/n_tok:.0f}%) are dynamic letters")
print("words with no dynamic letter:", " | ".join(w for w, v in hit.items() if not v))

dataset holds labelled with a dynamic letter: 317 of 1345 (24%)
letter    ن   م   ٹ   ئ   ہ   ج   آ   ھ  ش  ؤ  ڑ
holds   112  57  45  22  20  16  13  13  8  7  4

static dataset holds per signer: {'signer1-F': 234, 'signer2-F': 229, 'signer3-F': 217, 'signer4-L': 224, 'signer5-R': 124}

the 54 dictionary words: 44 contain at least one dynamic letter; 92 of 359 letter positions (26%) are dynamic letters
words with no dynamic letter: تخلیق | ذرات | راحت | سروسز | ضلع | طاقت | ظروف | غیر حاضر | ولید | ڈوسے


## 2. Split A: a new person, static letters

Same protocol as before: for each rotation a model is fitted on three signers and scored on the validation
signer, then refitted on four signers and scored on the test signer. Accuracy is the mean of the five signers;
macro recall is the mean of per-letter recall over the 29 static letters the dataset has.

The "40 letters" columns are the earlier notebook's numbers for the same model, for comparison. They are not
the same test: removing the dynamic letters removes 317 test holds as well as 11 classes.

In [3]:
ROT = splits.signer_rotations(sg)

def macro(ok, yy):
    return 100 * pd.Series(np.asarray(ok, float)).groupby(np.asarray(yy)).mean().mean()

def new_person(make, extra=None, seed=0):
    val = np.zeros(len(y)); t3 = np.zeros(len(y)); t4 = np.zeros(len(y))
    for train, va_s, te_s in ROT:
        tr = np.isin(sg, train); va = sg == va_s; te = sg == te_s
        def fit(rows):
            r = rows | extra if extra is not None else rows
            return make(seed).fit(P[r], y[r], groups=sg[r])
        m = fit(tr)
        val[va] = m.predict(P[va]) == y[va]; t3[te] = m.predict(P[te]) == y[te]
        t4[te] = fit(tr | va).predict(P[te]) == y[te]
    return val[D], t3[D], t4[D]

acc = lambda ok: 100 * ev.per_fold(ok, sD).mean()
base = ev.majority_correct(yD, sD)
print(f"majority baseline (most common training letter): {acc(base):.1f}%   chance for 29 letters: {100/29:.1f}%")

old = pd.read_csv(f"{OLD}/new_person.csv").set_index("model")
RES, rows = {}, []
def run(name, group, make, extra=None):
    t = time.time(); RES[name] = new_person(make, extra); v, t3, t4 = RES[name]
    rows.append({"model": name, "group": group, "val acc": acc(v), "val macro": macro(v, yD),
                 "test acc": acc(t4), "test macro": macro(t4, yD),
                 "40 letters: test acc": old.loc[name, "test acc (4 signers)"], "40 letters: test macro": old.loc[name, "test macro (4)"],
                 "seconds": round(time.time() - t)})

for name, (group, make) in lm.CANDIDATES.items():
    run(name, group, make)
for name in ("ExtraTrees", "ExtraTrees, rich features", "Neural net (PyTorch MLP)"):
    run(name + " + calibration takes", "new, more data", lm.CANDIDATES[name][1], extra=user)
TA = pd.DataFrame(rows).set_index("model")
TA.to_csv(f"{OUT}/new_person_static.csv")
print(TA.sort_values("test acc", ascending=False).round(1).to_string())

majority baseline (most common training letter): 13.0%   chance for 29 letters: 3.4%


                                                        group  val acc  val macro  test acc  test macro  40 letters: test acc  40 letters: test macro  seconds
model                                                                                                                                                         
5-model blend (geometric mean)                            new     37.9       31.7      41.3        35.6                  33.5                    27.2      120
ensemble: SVM+MLP+ExtraTrees+LDA                      earlier     38.3       28.6      40.8        30.6                  33.4                    23.1       69
ExtraTrees, rich features + calibration takes  new, more data     38.6       36.2      40.7        38.2                  32.1                    28.9       30
ExtraTrees, cleaned labels                                new     37.3       24.1      40.4        31.2                  33.1                    23.7       64
ExtraTrees + calibration takes                

### Is the gain from an easier task, or only from dropping hard holds?

Two things change at once when the dynamic letters go: the model has fewer letters to choose between, and the
holds of those letters leave the test. To separate them, the 40-letter model is scored on the static holds
only. Whatever it gains further when retrained on static letters is the benefit of not having the dynamic
classes to guess.

In [4]:
Pd, yd, sd = P0[~u0], M0.letter.to_numpy()[~u0], M0.signer.to_numpy()[~u0]
stat = ~np.isin(yd, splits.DYNAMIC_LETTERS)
drows = []
for name in ("SVM-RBF (shipped)", "ExtraTrees"):
    pred = np.empty(len(yd), dtype=object)
    for s in sorted(set(sd)):
        pred[sd == s] = lm.CANDIDATES[name][1](0).fit(Pd[sd != s], yd[sd != s], groups=sd[sd != s]).predict(Pd[sd == s])
    ok = (pred == yd).astype(float)
    drows.append({"model": name,
                  "40-letter model, all holds": 100 * ev.per_fold(ok, sd).mean(),
                  "40-letter model, static holds only": 100 * ev.per_fold(ok[stat], sd[stat]).mean(),
                  "static-letter model, static holds": acc(RES[name][2]),
                  "static holds the 40-letter model reads as a dynamic letter %": 100 * np.isin(pred[stat], splits.DYNAMIC_LETTERS).mean()})
print(pd.DataFrame(drows).set_index("model").round(1).to_string())

                   40-letter model, all holds  40-letter model, static holds only  static-letter model, static holds  static holds the 40-letter model reads as a dynamic letter %
model                                                                                                                                                                             
SVM-RBF (shipped)                        26.5                                29.6                               36.3                                                          23.1
ExtraTrees                               31.8                                37.5                               39.6                                                          10.9


### Choosing on validation, then testing; and each model against ExtraTrees

In [5]:
names = list(RES); ref = RES["ExtraTrees"][2]
def choose(criterion):
    ok = np.zeros(len(yD)); picked = {}
    for train, va_s, te_s in ROT:
        va, te = sD == va_s, sD == te_s
        score = {n: (RES[n][0][va].mean() if criterion == "accuracy" else macro(RES[n][0][va], yD[va])) for n in names}
        best = max(score, key=score.get); picked[te_s] = best
        ok[te] = RES[best][2][te]
    return ok, picked

sel_rows = []
for crit in ("accuracy", "macro recall"):
    ok, picked = choose(crit)
    r = ev.summarise(f"chosen by validation {crit}", ok, sD, baseline=ref)
    sel_rows.append({"how chosen": f"validation {crit}", "test accuracy %": r["mean of folds %"], "interval lo": r["boot lo %"],
                     "interval hi": r["boot hi %"], "test macro recall %": macro(ok, yD),
                     "vs ExtraTrees (pts)": r["vs baseline (pts)"], "paired p": r["paired t p"]})
    print(f"chosen by validation {crit}: " + "; ".join(f"{s}: {n}" for s, n in picked.items()))
for n in ("ExtraTrees", "SVM-RBF (shipped)"):
    r = ev.summarise(n, RES[n][2], sD, baseline=base)
    sel_rows.append({"how chosen": f"{n}, no choice", "test accuracy %": r["mean of folds %"], "interval lo": r["boot lo %"],
                     "interval hi": r["boot hi %"], "test macro recall %": macro(RES[n][2], yD)})
    print(f"{n} against the majority baseline: +{r['vs baseline (pts)']:.1f} pts, paired p = {r['paired t p']:.4f}")
print(); print(pd.DataFrame(sel_rows).round(3).to_string(index=False))

cmp_rows = []
for n in names:
    ok = RES[n][2]; d = ev.per_fold(ok - ref, sD); lo, hi = ev.bootstrap_by_signer(ok, sD)
    cmp_rows.append({"model": n, "test acc %": acc(ok), "interval": f"{100*lo:.1f} to {100*hi:.1f}", "macro recall %": macro(ok, yD),
                     "acc vs ExtraTrees": 100 * d.mean(), "wins of 5": int((d > 0).sum()),
                     "paired p": stats.ttest_1samp(d, 0).pvalue if n != "ExtraTrees" else np.nan,
                     "macro vs ExtraTrees": macro(ok, yD) - macro(ref, yD)})
CMP = pd.DataFrame(cmp_rows).set_index("model").sort_values("test acc %", ascending=False)
print(); print(CMP.round(3).to_string())

chosen by validation accuracy: signer1-F: ExtraTrees, rich features + calibration takes; signer2-F: ExtraTrees, rich features + calibration takes; signer3-F: kNN, k=15; signer4-L: ExtraTrees; signer5-R: 5-model blend (geometric mean)
chosen by validation macro recall: signer1-F: ExtraTrees, rich features + calibration takes; signer2-F: Neural net (PyTorch MLP) + calibration takes; signer3-F: ExtraTrees + calibration takes; signer4-L: ExtraTrees, rich features + calibration takes; signer5-R: LDA, rich features
ExtraTrees against the majority baseline: +26.7 pts, paired p = 0.0001


SVM-RBF (shipped) against the majority baseline: +23.4 pts, paired p = 0.0000

                  how chosen  test accuracy %  interval lo  interval hi  test macro recall %  vs ExtraTrees (pts)  paired p
         validation accuracy           40.429       37.158       43.229               33.669                0.793     0.402
     validation macro recall           40.286       37.872       42.222               35.856                0.650     0.393
       ExtraTrees, no choice           39.636       36.316       42.181               33.278                  NaN       NaN
SVM-RBF (shipped), no choice           36.326       35.040       37.397               30.109                  NaN       NaN



                                               test acc %      interval  macro recall %  acc vs ExtraTrees  wins of 5  paired p  macro vs ExtraTrees
model                                                                                                                                               
5-model blend (geometric mean)                     41.339  38.6 to 43.6          35.598              1.702          3     0.103                2.320
ensemble: SVM+MLP+ExtraTrees+LDA                   40.770  39.0 to 42.0          30.592              1.134          2     0.271               -2.686
ExtraTrees, rich features + calibration takes      40.668  38.1 to 43.1          38.171              1.032          4     0.324                4.893
ExtraTrees, cleaned labels                         40.358  38.4 to 42.4          31.210              0.722          4     0.420               -2.068
ExtraTrees + calibration takes                     39.896  35.3 to 43.2          37.264              0.26

### Per letter, for a new person

ExtraTrees and the shipped SVM on the test signers, static letters only.

In [6]:
pl = pd.DataFrame({"letter": yD, "ExtraTrees": RES["ExtraTrees"][2], "SVM": RES["SVM-RBF (shipped)"][2]}).groupby("letter")
PL = pd.DataFrame({"holds": pl.size(), "ExtraTrees recall": pl.ExtraTrees.mean().round(2), "SVM recall": pl.SVM.mean().round(2)})
print(PL.sort_values("ExtraTrees recall", ascending=False).T.to_string())

letter                ض     ط     ے      د      ک      س      ب      گ      ع      ڈ      و     ق       ر      ل      ف       ا       ی      ت     ث      ح    خ    ژ     ظ     غ    ذ    ص      ز    پ    چ
holds              10.0  4.00  15.0  13.00  93.00  52.00  85.00  21.00  13.00  49.00  77.00  10.0  120.00  75.00  14.00  130.00  134.00  19.00  4.00  26.00  5.0  5.0  10.0  10.0  5.0  4.0  12.00  5.0  8.0
ExtraTrees recall   0.8  0.75   0.6   0.54   0.52   0.50   0.49   0.48   0.46   0.45   0.40   0.4    0.40   0.39   0.36    0.36    0.31   0.26  0.25   0.23  0.2  0.2   0.2   0.1  0.0  0.0   0.00  0.0  0.0
SVM recall          0.7  0.75   0.4   0.38   0.53   0.44   0.52   0.14   0.31   0.41   0.45   0.4    0.38   0.35   0.36    0.28    0.30   0.21  0.25   0.12  0.0  0.4   0.2   0.1  0.2  0.0   0.17  0.0  0.0


## 3. Split B: a calibrated user, static letters

Training: the static dataset holds plus the training takes. Validation and test: calibration takes the model
has not seen, the same takes as in the earlier notebook (checked below). Mean of five random splits.

In [7]:
FILES = {"user-hassan": "2026-10-22", "user-hassan-session2": "2026-10-26", "user-hassan-session3": "2026-11-11",
         "user-hassan-session4": "2026-11-12", "user-mixed-abdullah-hassan-2026-11-13": "2026-11-13"}
frames = {}
for signer, d in FILES.items():
    for t in json.load(open(f"calibration/calibration_{d}.json", encoding="utf-8"))["takes"]:
        frames[(signer, f"calib_{t['letter']}_{t['take']}")] = np.array([f["pts"] for f in t["frames"]], dtype=np.float32)
FR = [frames[(s, v)] if u else None for s, v, u in zip(M.signer, M.video, user)]

def take_predict(m, idx):
    return np.array([m.classes_[m.predict_proba(FR[i]).mean(0).argmax()] for i in idx], dtype=object)

SPLITS = range(5)
for seed in SPLITS:                          # the same takes get the same role as before the letters were dropped
    a = splits.calibration_split(M, seed); b = splits.calibration_split(M0, seed)
    key = lambda MM, r: set(zip(MM.signer[r == "test"], MM.video[r == "test"]))
    assert key(M, a) == key(M0, b)
print("test takes identical to the earlier notebook's in all five splits")

CAL = {"SVM-RBF (shipped), browser vote share": lambda s=0: lm.SvmVoteShare(s),
       "SVM-RBF, Platt probabilities": lambda s=0: lm.OnFeatures("geom", models.svm(s))}
CAL.update({n: mk for n, (g, mk) in lm.CANDIDATES.items() if n not in ("SVM-RBF (shipped)", "ExtraTrees, cleaned labels")})
recs, PRED = [], {}
for name, make in CAL.items():
    t = time.time()
    for seed in SPLITS:
        role = splits.calibration_split(M, seed)
        tr = (~user) | (role == "train")
        m = make(0).fit(P[tr], y[tr], groups=sg[tr])
        for part in ("val", "test"):
            idx = np.where(role == part)[0]
            one = m.predict(P[idx]) == y[idx]; whole = take_predict(m, idx) == y[idx]
            PRED[(name, seed, part)] = (idx, one, whole)
            recs.append({"model": name, "split": seed, "part": part, "n": len(idx), "one frame %": 100 * one.mean(), "whole take %": 100 * whole.mean()})
    print(f"{name:40s} {time.time()-t:5.0f}s")
RB = pd.DataFrame(recs); RB.to_csv(f"{OUT}/calibrated_user_static.csv", index=False)

test takes identical to the earlier notebook's in all five splits


SVM-RBF (shipped), browser vote share       44s


SVM-RBF, Platt probabilities                43s


ExtraTrees                                 174s


MLP 128, alpha=3                            12s


LDA, shrinkage                               3s


... (8 outputs left out)

Neural net (PyTorch MLP), rich              74s


Graph network on the hand skeleton         128s


ExtraTrees + neural net                    155s


5-model blend (geometric mean)             161s


In [8]:
mean5 = lambda R: R[R.part == "test"].groupby("model", sort=False)[["one frame %", "whole take %"]].mean()
new, was = mean5(RB), mean5(pd.read_csv(f"{OLD}/calibrated_user.csv"))
B = pd.DataFrame({"one frame": new["one frame %"], "one frame, 41-class model": was["one frame %"],
                  "whole take": new["whole take %"], "whole take, 41-class model": was["whole take %"]})
val = RB[RB.part == "val"].groupby("model", sort=False)[["one frame %", "whole take %"]].mean()
B.insert(0, "val one frame", val["one frame %"]); B.insert(1, "val whole take", val["whole take %"])
print("Mean of five random splits, test takes (58 per split)")
print(B.sort_values("one frame", ascending=False).round(1).to_string())

print()
for measure, col in (("one frame", 1), ("whole take", 2)):
    picks, scores = [], []
    for seed in SPLITS:
        best = max(CAL, key=lambda n: (PRED[(n, seed, "val")][col].mean(), PRED[(n, seed, "val")][3 - col].mean()))
        picks.append(best); scores.append(100 * PRED[(best, seed, "test")][col].mean())
    print(f"{measure}: chosen on validation -> test {np.mean(scores):.1f}%  ({'; '.join(picks)})")

et = "ExtraTrees"; crow = []
for n in CAL:
    r = {"model": n}
    for measure, col in (("one frame", 1), ("whole take", 2)):
        a = np.concatenate([PRED[(n, s, "test")][col] for s in SPLITS]); b = np.concatenate([PRED[(et, s, "test")][col] for s in SPLITS])
        better, worse = int((a & ~b).sum()), int((~a & b).sum())
        r[f"{measure}: right of {len(a)}"] = int(a.sum())
        r[f"{measure}: better / worse than ExtraTrees"] = f"{better} / {worse}"
        r[f"{measure}: sign test p"] = stats.binomtest(better, better + worse).pvalue if better + worse else np.nan
    crow.append(r)
print(); print(pd.DataFrame(crow).set_index("model").sort_values("one frame: right of 290", ascending=False).round(3).to_string())

Mean of five random splits, test takes (58 per split)
                                       val one frame  val whole take  one frame  one frame, 41-class model  whole take  whole take, 41-class model
model                                                                                                                                             
Neural net (PyTorch MLP), rich                  96.2            97.2       96.6                       95.2        97.2                        95.5
Neural net (PyTorch MLP)                        95.2            96.6       94.8                       95.2        95.9                        95.9
ExtraTrees + neural net                         95.5            96.6       94.5                       95.5        96.2                        95.5
SVM-RBF (shipped), browser vote share           93.1            92.8       91.7                       87.6        92.8                        88.3
ExtraTrees                                      92.8            

### Which letters are still missed

In [9]:
show = ["SVM-RBF (shipped), browser vote share", "ExtraTrees", "Neural net (PyTorch MLP)"]
per = {}
for n in show:
    idx = np.concatenate([PRED[(n, s, "test")][0] for s in SPLITS]); ok = np.concatenate([PRED[(n, s, "test")][2] for s in SPLITS])
    g = pd.DataFrame({"letter": y[idx], "ok": ok}).groupby("letter").ok
    per[n] = g.sum().astype(int).astype(str) + "/" + g.size().astype(str); per[n + " missed"] = g.size() - g.sum()
L = pd.DataFrame(per)
L = L[L[[c for c in L if c.endswith("missed")]].sum(1) > 0]
print(L[[c for c in L if not c.endswith("missed")]].to_string())

       SVM-RBF (shipped), browser vote share ExtraTrees Neural net (PyTorch MLP)
letter                                                                          
ء                                       6/10       7/10                     7/10
ت                                       5/10       7/10                     5/10
ذ                                       9/10      10/10                    10/10
س                                       9/10      10/10                     9/10
ض                                       9/10      10/10                     9/10
ظ                                      10/10       9/10                    10/10
غ                                       8/10       7/10                     8/10
ق                                       9/10       9/10                    10/10
پ                                       5/10       5/10                    10/10
گ                                       9/10       6/10                    10/10


## 4. What this shows

**What the limitation costs.** 317 of the 1,345 dataset holds (24%) are dynamic letters, mostly ن (112), م (57)
and ٹ (45). 44 of the 54 dictionary words contain at least one, and 26% of all letter positions in those words are
dynamic letters. So the limitation is small for an alphabet demo and large for word recognition. The word decoder
was not re-run here.

**A new person, 29 static letters.**

| | accuracy | macro recall | on all 40 letters |
|---|---|---|---|
| majority baseline | 13.0% | | 10.0% |
| SVM (shipped recipe) | 36.3% | 30.1% | 26.5% / 23.7% |
| ExtraTrees | 39.6% | 33.3% | 31.8% / 27.3% |
| best new model (5-model blend) | 41.3% | 35.6% | 33.5% / 27.2% |
| model chosen on the validation signer | 40.4% (37.2 to 43.2) | 33.7% | 33.7% / 26.1% |

* Accuracy rises by about 8 points for ExtraTrees and 10 for the SVM. For ExtraTrees most of that (31.8 to 37.5)
  comes from the dynamic holds leaving the test, and 2 points (37.5 to 39.6) from no longer having those classes to
  guess. For the SVM the second part is larger (29.6 to 36.3): the 40-letter SVM reads 23% of static holds as a
  dynamic letter, ExtraTrees 11%.
* The ranking is unchanged. ExtraTrees is ahead of the SVM by 3.3 points on all five signers (p = 0.03). Nothing is
  reliably ahead of ExtraTrees: the best gain is 1.7 points (p = 0.10), and choosing on validation gives +0.8 (p = 0.40).
* The neural network and the graph network are still worse than ExtraTrees for a new person.
* Five static letters are never recognised for a new person by ExtraTrees: ذ ص ز پ چ (4 to 12 holds each).

**A calibrated user, 30 static letters** (mean of five splits, 58 test takes each; one frame / whole take):

| | 30-letter model | 41-letter model |
|---|---|---|
| SVM (shipped recipe, vote share) | 91.7% / 92.8% | 87.6% / 88.3% |
| ExtraTrees | 91.7% / 93.1% | 91.7% / 94.1% |
| PyTorch neural network | 94.8% / 95.9% | 95.2% / 95.9% |
| PyTorch neural network, rich features | 96.6% / 97.2% | 95.2% / 95.5% |
| model chosen on validation | 92.8% / 95.9% | 94.1% / 95.5% |

* **Retraining the shipped SVM on the 30 static letters gains about 4 points and brings it level with ExtraTrees**
  (6 takes better, 6 worse). This needs no change to the browser code, only a re-export of the weights.
* The neural network is still ahead: on rich features 16 takes better and 2 worse than ExtraTrees on single frames
  (p = 0.001), 15 against 3 on whole takes (p = 0.008). That model is the best of 17 on the test takes, so its own
  number is optimistic; the figure for a model picked on validation is 95.9% on whole takes.
* The scikit-learn MLP (alpha = 3) collapses to 27% here and pulls the four-model ensemble down with it. This is the
  setting, not a bug: scikit-learn scales the penalty by 1 / (number of rows), so with fewer training rows the same
  alpha is a stronger penalty and the network under-fits (33% on its own training rows). It was tuned on the 40-letter
  leave-one-signer-out data and should not be used on other data sizes without re-tuning.
* Remaining misses are mostly ت, ء, غ and پ (the SVM and ExtraTrees get پ right 5 times in 10; the neural network 10 in 10).

**Limits.** Same as the earlier notebook: the calibrated-user test takes come from the same sitting, people and room as
the training takes; the dataset labels are inferred, not checked; one seed per model; five signers. The candidate list
was shaped by exploration that saw 40-letter test scores.
